# UPCoM Model

In [ ]:
from pycontrails.datalib.ecmwf import ERA5
from pycontrails.datalib.emis_grid import AEIC
from pycontrails.models.upcom import UPCOM, UPCOMParams
from pycontrails.models.humidity_scaling import ConstantHumidityScaling
from pycontrails.models.sac import SAC
from pycontrails.models.issr import ISSR

from datetime import datetime

# For visualization
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

In [ ]:
# Load Gridded Emissions Data
aeic = AEIC(
    time=("2019-01", "2019-12"),
    variables=["FUELBURN", "BC", "DISTANCE"],
    data_dir="/Users/gett613/Data/AEIC/AEIC_monthly",
    pressure_levels=[85, 1020],
)
emis = aeic.open_metdataset()  # returns MetDataset
emis

In [ ]:
# Load meteorology data
time = (datetime(2022, 3, 1, 0), datetime(2022, 3, 1, 2))
variables = ["air_temperature", "specific_humidity"]
pressure_levels = [200, 250, 300]
era5 = ERA5(time, variables, pressure_levels)
met = era5.open_metdataset()
met

In [ ]:
## Quick and Dirty Interpolation. Convert the emission data do the meteorology grid
from pycontrails import MetDataset
import pandas as pd

# First interpolate the emission data onto the meteorology grid

emis_on_met_grid = MetDataset(
    emis.wrap_longitude().data.interp(
        longitude=met.data.longitude,
        latitude=met.data.latitude,
        level=met.data.level,
        method="linear",
    )
)

#emis_on_met_grid

# Then interpolate in time
met_years = pd.DatetimeIndex(met.data.time.values).year.unique()
if len(met_years) != 1:
    raise ValueError("Time interpolation currently requires met times within a single year.")
met_year = int(met_years[0])
relabeled_emis_times = pd.DatetimeIndex(
    [timestamp.replace(year=met_year) for timestamp in pd.DatetimeIndex(emis_on_met_grid.data.time.values)]
).to_numpy(dtype="datetime64[ns]")
emis_in_met_year = emis_on_met_grid.data.assign_coords(time=relabeled_emis_times)
emis_on_met_times = MetDataset(
    emis_in_met_year.interp(
        time=met.data.time,
        method="linear",
        kwargs={"bounds_error": True},
    )
)
np.testing.assert_array_equal(emis_on_met_times.data.time.values, met.data.time.values)
#emis_time_sample = emis_on_met_times.data.sel(longitude=0.0, latitude=45.0, method="nearest").compute()
#print(emis_time_sample)
emis_on_met_times

### Explore Biases in Horizontal Interpolation with Non-Conservative Mapping
There is non-conservative remapping.  

But it's not so bad. 

Code below takes the interpolated emissions at a level, 
intepolates the original emissions to the same vertical level with the same method, and computes the errors and weighted total bias


In [ ]:
import pandas as pd
import xarray as xr
from pycontrails.physics import constants


def centered_global_cell_area(dataset):
    longitude = dataset.longitude.values
    latitude = dataset.latitude.values
    longitude_spacing = np.diff(longitude)
    if not (
        np.allclose(longitude_spacing, longitude_spacing[0])
        and np.isclose(longitude_spacing[0] * longitude.size, 360.0)
        and np.all(np.diff(latitude) > 0)
        and np.isclose(latitude[0], -90.0)
        and np.isclose(latitude[-1], 90.0)
    ):
        raise ValueError("This diagnostic requires global grids with regular, nonduplicated longitudes and pole-to-pole latitudes.")
    latitude_edges = np.concatenate(([-90.0], (latitude[:-1] + latitude[1:]) / 2, [90.0]))
    latitude_weights = np.diff(np.sin(np.deg2rad(latitude_edges)))
    areas = constants.radius_earth**2 * np.deg2rad(longitude_spacing[0]) * latitude_weights
    return xr.DataArray(
        np.broadcast_to(areas, (longitude.size, latitude.size)),
        dims=("longitude", "latitude"),
        coords={"longitude": longitude, "latitude": latitude},
    )


source_emissions = emis.data
mapped_emissions = emis_on_met_grid.data
source_area = centered_global_cell_area(source_emissions)
mapped_area = centered_global_cell_area(mapped_emissions)
np.testing.assert_allclose(source_area.sum(), mapped_area.sum(), rtol=1e-12)
source_at_target_levels = source_emissions.interp(level=mapped_emissions.level)
source_at_target_levels, mapped_emissions = xr.align(
    source_at_target_levels, mapped_emissions, join="exact", exclude={"longitude", "latitude"}
)

remap_error_tables = []
for variable_name in mapped_emissions.data_vars:
    source_field = source_at_target_levels[variable_name].reset_coords(drop=True)
    mapped_field = mapped_emissions[variable_name].reset_coords(drop=True)
    source_total = (source_field.where(np.isfinite(source_field)) * source_area).sum(
        dim=["longitude", "latitude"], skipna=False
    )
    mapped_total = (mapped_field.where(np.isfinite(mapped_field)) * mapped_area).sum(
        dim=["longitude", "latitude"], skipna=False
    )
    diagnostics = xr.Dataset({
        "source_total": source_total,
        "mapped_total": mapped_total,
        "change_pct": 100 * (mapped_total - source_total) / source_total.where(source_total != 0),
        "source_missing_area_pct": 100 * (source_area * ~np.isfinite(source_field)).sum(
            dim=["longitude", "latitude"]
        ) / source_area.sum(),
        "mapped_missing_area_pct": 100 * (mapped_area * ~np.isfinite(mapped_field)).sum(
            dim=["longitude", "latitude"]
        ) / mapped_area.sum(),
    }).compute()
    variable_table = diagnostics.to_dataframe().reset_index()
    variable_table["variable"] = variable_name
    variable_table["total_units"] = "km/s" if variable_name == "DISTANCE" else "kg/s"
    remap_error_tables.append(variable_table)

remap_error_table = pd.concat(remap_error_tables, ignore_index=True)
month_seconds = pd.to_datetime(remap_error_table["time"]).dt.days_in_month * 86400
remap_error_table["source_month_total"] = remap_error_table["source_total"] * month_seconds
remap_error_table["mapped_month_total"] = remap_error_table["mapped_total"] * month_seconds
remap_error_summary = remap_error_table.groupby(["variable", "level"]).agg(
    mean_change_pct=("change_pct", "mean"),
    min_change_pct=("change_pct", "min"),
    max_change_pct=("change_pct", "max"),
    max_source_missing_area_pct=("source_missing_area_pct", "max"),
    max_mapped_missing_area_pct=("mapped_missing_area_pct", "max"),
)
annual_totals = remap_error_table.groupby(["variable", "level"])[
    ["source_month_total", "mapped_month_total"]
].agg(lambda values: values.sum(skipna=False))
remap_error_summary["annual_bias_pct"] = (
    100 * (annual_totals["mapped_month_total"] - annual_totals["source_month_total"])
    / annual_totals["source_month_total"].where(annual_totals["source_month_total"] != 0)
)
print("Horizontal remapping: area-integrated rate change at each target level, over all months.")
print("mean_change_pct: signed mean of monthly percentages; annual_bias_pct: change in totals weighted by month duration (monthly-mean rates assumed).")
print("Positive change = gain; negative change = loss. Missing data invalidate totals; zero source totals give undefined percentages.")
print("This does not measure vertical conservation or local redistribution error.")
print(remap_error_summary.to_string(float_format=lambda value: f"{value:.6f}"))
remap_error_table

In [ ]:
# Create model with optional humidity scaling
scaling = ConstantHumidityScaling(rhi_adj=0.98)
model = UPCOM(met, humidity_scaling=scaling)

In [ ]:
# Run evaluation
result = model.eval()

In [ ]:
result

In [ ]:
# Access outputs
rhi = result["rhi"]
issr = result["issr"]
persistent_contrails = result["potential_persistent_contrail"]
temp = result["air_temperature"]
sac = result["sac"]

In [ ]:
# Get comparison from Native pycontrails routines (code from pycontrails/notebooks/SAC.ipynb and ISSR.ipynb)

# run model for across full input domain
# outputs global ice super-saturated regions as 1.0, all other regions as 0.0
issr_mds = ISSR(met, humidity_scaling=scaling).eval()
issr_default = issr_mds["issr"]

# evaluate SAC on met grid
sac_mds = SAC(met=met, humidity_scaling=scaling).eval()  # returns a MetDataset
sac_default = sac_mds["sac"]  # extract the SAC MetDataArray

# PPCF occurs where both the ISSR and SAC criteria are satisfied
ppc_default = (issr_default.data == 1) & (sac_default.data == 1)

### Comparison Stats

In [ ]:
comparison = "ppc"
comparison_fields = {
    "issr": (issr.data, issr_default.data),
    "sac": (sac.data, sac_default.data),
    "ppc": (persistent_contrails.data, ppc_default),
}
if comparison not in comparison_fields:
    raise ValueError("comparison must be 'issr', 'sac', or 'ppc'")

upcom_field, reference_field = xr.align(*comparison_fields[comparison], join="exact")
difference_mask = (upcom_field != reference_field) & ~(
    upcom_field.isnull() & reference_field.isnull()
)
difference_counts = difference_mask.sum(dim=["latitude", "longitude"]).compute()

print(f"{comparison.upper()}: UPCoM vs native pycontrails")
for level in difference_counts.level.values:
    for valid_time in difference_counts.time.values:
        count = int(difference_counts.sel(level=level, time=valid_time).item())
        time_label = np.datetime_as_string(valid_time, unit="h")
        print(f"Level {level:g} hPa | {time_label} | differences: {count if count else 'none'}")

In [ ]:
import cartopy.crs as ccrs

plot_level = 250
time_index = 0

selected_difference_mask = (
    difference_mask.sel(level=plot_level)
    .isel(time=time_index)
    .transpose("latitude", "longitude")
    .compute()
)
latitude_indices, longitude_indices = np.nonzero(selected_difference_mask.values)
difference_count = latitude_indices.size
plot_time_label = np.datetime_as_string(selected_difference_mask.time.values, unit="h")

fig, ax = plt.subplots(
    figsize=(12, 6),
    subplot_kw={"projection": ccrs.Robinson(central_longitude=0)},
)
ax.set_global()
ax.coastlines(linewidth=0.8)
ax.gridlines(alpha=0.4, linestyle="--")
ax.scatter(
    selected_difference_mask.longitude.values[longitude_indices],
    selected_difference_mask.latitude.values[latitude_indices],
    transform=ccrs.PlateCarree(),
    s=20,
    color="crimson",
    linewidths=0,
    label="Different grid locations",
    zorder=3,
)
if difference_count:
    ax.legend(loc="lower left")
else:
    ax.text(0.5, 0.5, "No differences", transform=ax.transAxes, ha="center", va="center")
ax.set_title(
    f"{comparison.upper()} difference mask | {plot_level} hPa | {plot_time_label}\n"
    f"UPCoM vs native pycontrails: {difference_count} different locations"
)
plt.tight_layout()
plt.show()

In [ ]:
sac_default.data.sel(time=result.data.time[0], level=250).transpose().plot(figsize=(12, 6))
plt.title('SAC from pycontrails at 250 hPa')
plt.show()

In [ ]:
ppc_default.sel(time=result.data.time[0], level=250).transpose().plot(
    figsize=(12, 6), cbar_kwargs={"label": "PPCF (0/1)"}
)
plt.title('Potential persistent contrail formation at 250 hPa')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

level = 200
time_value = result.data.time[0]
model_issr = result.data["issr"].sel(time=time_value, level=level)
reference_issr = issr_default.data.sel(time=time_value, level=level)
issr_difference = model_issr - reference_issr

fig, axes = plt.subplots(
    3, 1, figsize=(12, 15),
    subplot_kw={"projection": ccrs.Robinson(central_longitude=0)},
)
panel_data = [model_issr, reference_issr, issr_difference]
panel_titles = ["(a) UPCOM result['issr']", "(b) ISSR", "(c) Difference (UPCOM - ISSR)"]
panel_cmaps = ["YlOrRd", "YlOrRd", "RdBu_r"]
panel_limits = [(0, 1), (0, 1), (-1, 1)]
panel_cbar_labels = ["ISSR (0/1)", "ISSR (0/1)", "Difference (UPCOM - ISSR)"]

for ax, data, title, cmap, (vmin, vmax), cbar_label in zip(
    axes, panel_data, panel_titles, panel_cmaps, panel_limits, panel_cbar_labels
):
    data.transpose().plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        add_labels=False,
        cbar_kwargs={
            "orientation": "horizontal",
            "pad": 0.05,
            "shrink": 0.75,
            "label": cbar_label,
        },
    )
    ax.coastlines(linewidth=0.8)
    ax.gridlines(alpha=0.4, linestyle="--")
    ax.set_global()
    ax.set_title(title)

fig.suptitle(f"ISSR comparison at {level} hPa", y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
##var='air_temperature'
#var='potential_persistent_contrail'
##var='T_contr'
#result.data[var].sel(time=result.data.time[0], level=250).transpose().plot(figsize=(12, 6))
#plt.title(var+' at 250 hPa')
#plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

level = 300
time_value = result.data.time[1]
model_ppcf = result.data["potential_persistent_contrail"].sel(
    time=time_value, level=level
)
reference_ppcf = ppc_default.sel(time=time_value, level=level)
ppcf_difference = model_ppcf - reference_ppcf

fig, axes = plt.subplots(
    3, 1, figsize=(12, 15),
    subplot_kw={"projection": ccrs.Robinson(central_longitude=0)},
)
panel_data = [model_ppcf, reference_ppcf, ppcf_difference]
panel_titles = [
    "(a) UPCOM result['potential_persistent_contrail']",
    "(b) PPCF (ISSR & SAC)",
    "(c) Difference (UPCOM - PPCF)",
]
panel_cmaps = ["YlOrRd", "YlOrRd", "RdBu_r"]
panel_limits = [(0, 1), (0, 1), (-1, 1)]
panel_cbar_labels = [
    "Potential persistent contrail (0/1)",
    "PPCF (0/1)",
    "Difference (UPCOM - PPCF)",
]

for ax, data, title, cmap, (vmin, vmax), cbar_label in zip(
    axes, panel_data, panel_titles, panel_cmaps, panel_limits, panel_cbar_labels
):
    data.transpose().plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        add_labels=False,
        cbar_kwargs={
            "orientation": "horizontal",
            "pad": 0.05,
            "shrink": 0.75,
            "label": cbar_label,
        },
    )
    ax.coastlines(linewidth=0.8)
    ax.gridlines(alpha=0.4, linestyle="--")
    ax.set_global()
    ax.set_title(title)

fig.suptitle(f"PPCF comparison at {level} hPa", y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

level = 250
time_value = result.data.time[1]
model_sac = result["sac"].data.sel(time=time_value, level=level)
reference_sac = sac_default.data.sel(time=time_value, level=level)
sac_difference = model_sac - reference_sac

fig, axes = plt.subplots(
    3, 1, figsize=(12, 15),
    subplot_kw={"projection": ccrs.Robinson(central_longitude=0)},
)
panel_data = [model_sac, reference_sac, sac_difference]
panel_titles = [
    "(a) UPCOM result['sac']",
    "(b) SAC default",
    "(c) Difference (UPCOM - SAC default)",
]
panel_cmaps = ["YlOrRd", "YlOrRd", "RdBu_r"]
panel_limits = [(0, 1), (0, 1), (-1, 1)]
panel_cbar_labels = [
    "SAC (0/1)",
    "SAC default (0/1)",
    "Difference (UPCOM - SAC default)",
]

for ax, data, title, cmap, (vmin, vmax), cbar_label in zip(
    axes, panel_data, panel_titles, panel_cmaps, panel_limits, panel_cbar_labels
):
    data.transpose().plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        add_labels=False,
        cbar_kwargs={
            "orientation": "horizontal",
            "pad": 0.05,
            "shrink": 0.75,
            "label": cbar_label,
        },
    )
    ax.coastlines(linewidth=0.8)
    ax.gridlines(alpha=0.4, linestyle="--")
    ax.set_global()
    ax.set_title(title)

fig.suptitle(f"SAC comparison at {level} hPa", y=0.98)
plt.tight_layout()
plt.show()

### Plot

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, axes = plt.subplots(2, 2, figsize=(16, 8), 
                         subplot_kw={'projection': ccrs.PlateCarree()})

level = 300
time_idx = 1

fields = ['rhi', 'rh_liquid', 'issr', 'potential_persistent_contrail']
titles = ['RHi', 'RH (liquid)', 'ISSR', 'Persistent Contrails']

for ax, field, title in zip(axes.flat, fields, titles):
    data = result.data[field].sel(time=result.data.time[time_idx], level=level)
    data.transpose().plot(ax=ax, transform=ccrs.PlateCarree(), cbar_kwargs={'shrink': 0.8})
    ax.coastlines(color="white")
    ax.gridlines(alpha=0.8, linestyle='--')
    ax.set_title(f'{title} at {level} hPa')

plt.tight_layout()
plt.show()

### Not sure these work....

In [ ]:
### Uses built in xarray plotting

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
#import cartopy.feature as cfeature

fig, axes = plt.subplots(2, 2, figsize=(20, 14), 
                         subplot_kw={'projection': ccrs.Robinson()})

level = 250
time_idx = 0

fields = ['rhi', 'rh_liquid', 'issr', 'potential_persistent_contrail']
titles = ['Relative Humidity over Ice', 'Relative Humidity over Liquid', 
          'Ice Supersaturated Regions', 'Potential Persistent Contrails']
cmaps = ['RdYlBu_r', 'RdYlBu_r', 'YlOrRd', 'RdYlGn']
vmins = [0, 0, 0, 0]
vmaxs = [1.5, 1, 1, 1]

for ax, field, title, cmap, vmin, vmax in zip(axes.flat, fields, titles, cmaps, vmins, vmaxs):
    data = result.data[field].sel(time=result.data.time[time_idx], level=level)
    
    # Use xarray's plot with transform
    data.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap, vmin=vmin, vmax=vmax,
        cbar_kwargs={'orientation': 'horizontal', 'pad': 0.05, 
                     'shrink': 0.8, 'label': title}
    )
    
    ax.coastlines(linewidth=0.8)
#    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':', alpha=0.5)
    ax.gridlines(alpha=0.4, linestyle='--')
    ax.set_global()
    ax.set_title(title, fontsize=12, fontweight='bold')

fig.suptitle(f'UPCOM Model Output at {level} hPa\n{result.data.time[time_idx].values}', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
#Uses contourf (matplotlib)
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
#import cartopy.feature as cfeature

fig, axes = plt.subplots(2, 2, figsize=(20, 14), 
                         subplot_kw={'projection': ccrs.Robinson(central_longitude=0)})

level = 250
time_idx = 0

fields = ['rhi', 'rh_liquid', 'issr', 'potential_persistent_contrail']
titles = ['Relative Humidity over Ice', 'Relative Humidity over Liquid', 
          'Ice Supersaturated Regions', 'Potential Persistent Contrails']
cmaps = ['RdYlBu_r', 'RdYlBu_r', 'YlOrRd', 'RdYlGn']
vmins = [0, 0, 0, 0]
vmaxs = [1.5, 1, 1, 1]

for ax, field, title, cmap, vmin, vmax in zip(axes.flat, fields, titles, cmaps, vmins, vmaxs):
    data = result.data[field].sel(time=result.data.time[time_idx], level=level)
    
    # Use contourf instead of pcolormesh
    im = ax.contourf(
        data.longitude, data.latitude, data.values,
        levels=20,  # Number of contour levels
        transform=ccrs.PlateCarree(),
        cmap=cmap, vmin=vmin, vmax=vmax,
        extend='both'
    )
    
    ax.coastlines(linewidth=0.8, color='black')
#    ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
#    ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.2)
#    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':', alpha=0.5)
    ax.gridlines(alpha=0.4, linestyle='--')
    ax.set_global()
    
    cbar = plt.colorbar(im, ax=ax, orientation='horizontal', 
                       pad=0.05, shrink=0.8, aspect=30)
    cbar.set_label(title, fontsize=10)
    
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)

fig.suptitle(f'UPCOM Model Output at {level} hPa\n{result.data.time[time_idx].values}', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


### Diagnostics for Debugging

In [ ]:
import numpy as np

# Diagnostic code
print("=== Diagnostics ===")
print(f"T_contr type: {type(result.data['T_contr'])}")
print(f"T_contr shape: {result.data['T_contr'].shape}")
print(f"T_contr dims: {result.data['T_contr'].dims}")
print(f"T_contr min/max: {result.data['T_contr'].min().values:.2f} / {result.data['T_contr'].max().values:.2f}")

print(f"\nAir temp min/max: {result.data['air_temperature'].min().values:.2f} / {result.data['air_temperature'].max().values:.2f}")
print(f"RHi min/max: {result.data['rhi'].min().values:.3f} / {result.data['rhi'].max().values:.3f}")
print(f"RH_liquid min/max: {result.data['rh_liquid'].min().values:.3f} / {result.data['rh_liquid'].max().values:.3f}")

print(f"\nG min/max: {result.data['G'].min().values:.6f} / {result.data['G'].max().values:.6f}")
print(f"G > 0.053: {(result.data['G'] > 0.053).sum().values} points")

print(f"\nCondition checks (count of True):")
print(f"  RHi > 1.0: {(result.data['rhi'] > 1.0).sum().values}")
print(f"  T < T_contr: {(result.data['air_temperature'] < result.data['T_contr']).sum().values}")
print(f"  RH_liquid > RH_contr: {(result.data['rh_liquid'] > result.data['RH_contr']).sum().values}")

print(f"\nPersistent contrails: {result.data['potential_persistent_contrail'].sum().values}")

# Check a specific point where RHi > 1
idx = np.where(result.data['rhi'].values > 1.0)
if len(idx[0]) > 0:
    i = idx[0][0]
    print(f"\n=== Example point with RHi > 1 ===")
    print(f"RHi: {result.data['rhi'].values.flat[i]:.3f}")
    print(f"T: {result.data['air_temperature'].values.flat[i]:.2f} K")
    print(f"T_contr: {result.data['T_contr'].values.flat[i]:.2f} K")
    print(f"RH_liquid: {result.data['rh_liquid'].values.flat[i]:.3f}")
    print(f"RH_contr: {result.data['RH_contr'].values.flat[i]:.3f}")
    print(f"G: {result.data['G'].values.flat[i]:.6f}")



In [ ]:
print(f"Air pressure min/max: {result.data['air_pressure'].min().values} / {result.data['air_pressure'].max().values}")
print(f"Air pressure units: {result.data['air_pressure'].attrs.get('units', 'not specified')}")
print(f"Air pressure shape: {result.data['air_pressure'].shape}")
print(f"Air pressure dims: {result.data['air_pressure'].dims}")

# Also check a single value
print(f"\nSample pressure value: {result.data['air_pressure'].values.flat[0]}")

In [ ]:
from pycontrails.physics import constants
print(f"c_pd = {constants.c_pd}")
print(f"epsilon = {constants.epsilon}")

In [ ]:
print(f"air_pressure dims: {met.data['air_pressure'].dims}")
print(f"air_pressure coords: {met.data['air_pressure'].coords}")
print(f"air_temperature dims: {met.data['air_temperature'].dims}")